In [6]:
import numpy as np 
import pandas as pd 
import os 
import glob
import warnings
warnings.filterwarnings('ignore')    ## I don't like pandas setting with copy warnings 
from scipy.stats import wilcoxon


In [3]:
def process_csvs(input_folder, output_folder):
    """
    Reads all CSVs from input_folder, removes rows where iso3 or Country == 'EMR',
    and saves them to output_folder with the same filename.
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all CSV files in input folder
    csv_files = glob.glob(os.path.join(input_folder, '*.csv'))
    
    for csv_file in csv_files:
        # Read the CSV
        df = pd.read_csv(csv_file)
        
        # Remove rows where iso3 or Country equals 'EMR'
        if 'iso3' in df.columns:
            df = df[df['iso3'] != 'EMR']
        if 'Country' in df.columns:
            df = df[df['Country'] != 'EMR']
        
        # Get filename and save to output folder
        filename = os.path.basename(csv_file)
        output_path = os.path.join(output_folder, filename)
        df.to_csv(output_path, index=False)
        print(f"Processed and saved: {filename}")

# Usage
process_csvs(input_folder='your_input_folder', output_folder='your_output_folder')

#in emissions v3 i have removed 1990 IRQ because of the incorrect unprocessed red meat data for it which was much higher than expected (174 g/d)... but there are EMR means included in the csvs which i should remove now
process_csvs(input_folder=r'co2_calc\emissions_v3', output_folder='co2_calc\emissions_v5')

Processed and saved: Beans_and_legumes_all.csv
Processed and saved: Beans_and_legumes_females.csv
Processed and saved: Beans_and_legumes_males.csv
Processed and saved: Cheese_all.csv
Processed and saved: Cheese_females.csv
Processed and saved: Cheese_males.csv
Processed and saved: Eggs_all.csv
Processed and saved: Eggs_females.csv
Processed and saved: Eggs_males.csv
Processed and saved: Fruits_all.csv
Processed and saved: Fruits_females.csv
Processed and saved: Fruits_males.csv
Processed and saved: Fruit_juices_all.csv
Processed and saved: Fruit_juices_females.csv
Processed and saved: Fruit_juices_males.csv
Processed and saved: Milk_all.csv
Processed and saved: Milk_females.csv
Processed and saved: Milk_males.csv
Processed and saved: Non_starchy_vegetables_all.csv
Processed and saved: Non_starchy_vegetables_females.csv
Processed and saved: Non_starchy_vegetables_males.csv
Processed and saved: Nuts_all.csv
Processed and saved: Nuts_females.csv
Processed and saved: Nuts_males.csv
Process

In [8]:

def test_regional_vs_global(
    regional_path: str, 
    global_path: str, 
    value_col: str = 'emissions', 
    time_col: str = 'year', 
    years_to_drop: list = None
):
    """
    Reads regional and global time-series data, drops specific years (e.g., pandemic anomalies),
    calculates the regional median per year, and runs a Wilcoxon signed-rank test.
    """
    reg_df = pd.read_csv(regional_path)
    # reg_df = reg_df[reg_df['Country'] != 'EMR'] 
    glob_df = pd.read_csv(global_path)

    if years_to_drop:
        reg_df = reg_df[~reg_df[time_col].isin(years_to_drop)]
        glob_df = glob_df[~glob_df[time_col].isin(years_to_drop)]

    reg_med = reg_df.groupby(time_col).agg({value_col: 'mean'}).reset_index()
    reg_med = reg_med.rename(columns={value_col: 'regional_mean'})

    glob_clean = glob_df[[time_col, value_col]].rename(columns={value_col: 'global_value'})

    merged_df = pd.merge(reg_med, glob_clean, on=time_col, how='inner')

    if merged_df.empty:
        raise ValueError("Merged dataframe is empty. Check column names and time overlaps.")

    stat, p_value = wilcoxon(merged_df['regional_mean'], merged_df['global_value'])
    
    print(f"--- Results for {value_col.upper()} ---")
    print(f"Years analyzed: {len(merged_df)} paired timepoints")
    print(f"Wilcoxon statistic: {stat:.4f}, p-value: {p_value:.4f}\n")

    return stat, p_value, merged_df


In [12]:
stat, p_val, paired_data = test_regional_vs_global(
    regional_path=r'co2_calc\emissions_v5\total_all.csv',
    global_path=r'co2_calc\global_emissions\total_all.csv',
    years_to_drop=[2020]
)
paired_data.round(1)

--- Results for EMISSIONS ---
Years analyzed: 7 paired timepoints
Wilcoxon statistic: 0.0000, p-value: 0.0156



,year,regional_mean,global_value
0,1990,2038.1,1928.7
1,1995,2221.4,1947.0
2,2000,2282.9,2015.3
3,2005,2335.5,2117.6
4,2010,2455.3,2290.2
5,2015,2442.1,2280.9
6,2018,2369.9,2309.4


In [13]:
stat, p_val, paired_data = test_regional_vs_global(
    regional_path=r'co2_calc\emissions_v5\total_males.csv',
    global_path=r'co2_calc\global_emissions\total_males.csv',
    years_to_drop=[2020]
)
paired_data.round(1)

--- Results for EMISSIONS ---
Years analyzed: 7 paired timepoints
Wilcoxon statistic: 0.0000, p-value: 0.0156



,year,regional_mean,global_value
0,1990,2061.3,1938.8
1,1995,2246.2,1960.3
2,2000,2311.7,2030.5
3,2005,2364.2,2129.7
4,2010,2485.0,2302.7
5,2015,2467.7,2293.6
6,2018,2394.8,2322.8


In [14]:
stat, p_val, paired_data = test_regional_vs_global(
    regional_path=r'co2_calc\emissions_v5\total_females.csv',
    global_path=r'co2_calc\global_emissions\total_females.csv',
    years_to_drop=[2020]
)
paired_data.round(1)

--- Results for EMISSIONS ---
Years analyzed: 7 paired timepoints
Wilcoxon statistic: 0.0000, p-value: 0.0156



,year,regional_mean,global_value
0,1990,1998.9,1918.8
1,1995,2179.0,1933.6
2,2000,2238.8,1998.7
3,2005,2291.4,2103.0
4,2010,2406.4,2276.6
5,2015,2395.1,2265.8
6,2018,2323.5,2294.3


In [20]:
import pandas as pd
from scipy.stats import wilcoxon

def run_comparison_co2(
    csv1_path: str, 
    csv2_path: str, 
    time_col: str, 
    country_col: str,
    drop_years: list = None,
    value_col: str = 'emissions' # Defaulting to your column name
):
    # 1. Load data
    df1 = pd.read_csv(csv1_path)
    df2 = pd.read_csv(csv2_path)

    # 2. Drop years if specified
    if drop_years:
        df1 = df1[~df1[time_col].isin(drop_years)]
        df2 = df2[~df2[time_col].isin(drop_years)]

    # 3. Merge on both time and country columns
    merged = pd.merge(
        df1, 
        df2, 
        on=[time_col, country_col], 
        how='inner', 
        suffixes=('_emr_males', '_emr_females')
    )
    col1, col2 = f'{value_col}_emr_males', f'{value_col}_emr_females'

    # 4. Safety check
    if merged.empty:
        raise ValueError("Merged dataframe is empty. Check your CSVs for overlapping times and countries.")

    # 5. Drop any rows where the target values might be NaN after merge (optional but recommended for Wilcoxon)
    merged = merged.dropna(subset=[col1, col2])

    # 6. Run the Wilcoxon paired test
    stat, p_val = wilcoxon(merged[col1], merged[col2])
    
    # 7. Output
    print("--- TEST SETTING: EMR VS EMR ---")
    print(f"Paired data points analyzed: {len(merged)}")
    print(f"Wilcoxon statistic: {stat:.4f}, p-value: {p_val:.4f}\n")

    return stat, p_val, merged

In [23]:
stat, p, data = run_comparison_co2(
    csv1_path=r'co2_calc\emissions_v5\total_males.csv',
    csv2_path=r'co2_calc\emissions_v5\total_females.csv',
    time_col='year',
    country_col='Country',
    drop_years=[2020]
)

data.round(1)

--- TEST SETTING: EMR VS EMR ---
Paired data points analyzed: 146
Wilcoxon statistic: 1586.0000, p-value: 0.0000



,Country,year,emissions_emr_males,emissions_emr_females
0,AFG,1990,1871.9,1871.1
1,AFG,1995,1606.6,1585.5
2,AFG,2000,1543.9,1512.3
3,AFG,2005,1431.5,1404.5
4,AFG,2010,1389.8,1359.4
...,...,...,...,...
141,YEM,2000,1144.6,1147.7
142,YEM,2005,1445.9,1441.5
143,YEM,2010,1425.5,1412.1
144,YEM,2015,1633.3,1614.8


In [59]:
def analyze_wide_csvs(
    setting: str, 
    csv1_path: str, 
    csv2_path: str, 
    name1: str,  
    name2: str,  
    id_column: str = 'iso3'
):
    # 1. Read files
    df1 = pd.read_csv(csv1_path)
    df2 = pd.read_csv(csv2_path)

    # Standardize column names in df2 to lowercase to safely catch 'year'
    df2.columns = [str(c).strip().lower() for c in df2.columns]
    id_column_lower = id_column.lower()

    # 2. Inject missing ID column for global data if needed
    if setting == 'emr vs global' and id_column_lower not in df2.columns:
        df2[id_column_lower] = 'Global_Placeholder'

    # 3. Reshape from Wide to Long (Isolating ONLY digit columns)
    # --- DF1 Processing ---
    years1 = [col for col in df1.columns if str(col).isdigit()]
    if years1:
        long1 = df1.melt(id_vars=[id_column], value_vars=years1, var_name='year', value_name=name1)
    else:
        long1 = df1.copy()
        # If already long, rename the non-year/non-id column to name1
        val_cols = [c for c in long1.columns if str(c).strip().lower() not in ['year', id_column_lower]]
        if val_cols:
            long1 = long1.rename(columns={val_cols[0]: name1})

    # --- DF2 Processing (Handles the 'year, score' global format) ---
    years2 = [col for col in df2.columns if str(col).isdigit()]
    if years2:
        long2 = df2.melt(id_vars=[id_column_lower], value_vars=years2, var_name='year', value_name=name2)
    else:
        long2 = df2.copy()
        # Identify the value column (e.g., 'score') and rename it to name2
        val_cols = [c for c in long2.columns if c not in ['year', id_column_lower]]
        if val_cols:
            long2 = long2.rename(columns={val_cols[0]: name2})

    # 4. Clean empty values (like the trailing 2020 comma)
    # This drops the 2020 row automatically because pandas parses '2020,' as NaN for the value column.
    long1 = long1.dropna(subset=[name1])
    long2 = long2.dropna(subset=[name2])
    
    # Format years safely
    # Ensure 'year' column exists in both and convert cleanly
    year_col1 = [c for c in long1.columns if str(c).strip().lower() == 'year'][0]
    year_col2 = [c for c in long2.columns if str(c).strip().lower() == 'year'][0]

    long1['year'] = long1[year_col1].astype(str).str.strip().astype(int)
    long2['year'] = long2[year_col2].astype(str).str.strip().astype(int)

    # 5. Route logic based on setting
    if setting == 'emr vs global':
        ready1 = long1.groupby('year').agg({name1: 'mean'}).reset_index()
        ready2 = long2[['year', name2]]
        group1_name, group2_name = f'{name1} Median', f'{name2} Value'

    elif setting == 'emr vs emr':
        ready1 = long1.groupby('year').agg({name1: 'mean'}).reset_index()
        ready2 = long2.groupby('year').agg({name2: 'mean'}).reset_index()
        group1_name, group2_name = f'{name1} Median', f'{name2} Median'
        
    else:
        raise ValueError("Argument 'setting' must be either 'emr vs global' or 'emr vs emr'")

    # 6. Merge on time to ensure strict pairing
    merged = pd.merge(ready1, ready2, on='year', how='inner')

    if merged.empty:
        raise ValueError("Merged dataframe is empty. Check overlapping years.")

    # 7. Run the Wilcoxon test
    stat, p_val = wilcoxon(merged[name1], merged[name2])

    # 8. Output Results
    print(f"--- TEST SETTING: {setting.upper()} ---")
    print(f"Comparison: {group1_name} vs {group2_name}")
    print(f"Paired timepoints analyzed: {len(merged)} years")
    print(f"Wilcoxon statistic: {stat:.4f}, p-value: {p_val:.4f}\n")

    return stat, p_val, merged

**PHDI Comparison**

In [60]:
stat, p, paired_data = analyze_wide_csvs(
    setting='emr vs global',
    csv1_path=r'diet_calc\scores\phdi\total_all.csv',
    csv2_path=r'diet_calc\scores\phdi_global\total_all.csv',
    name1='EMR PHDI',
    name2='Global PHDI',
    id_column='Country'
)
paired_data.round(1)

--- TEST SETTING: EMR VS GLOBAL ---
Comparison: EMR PHDI Median vs Global PHDI Value
Paired timepoints analyzed: 7 years
Wilcoxon statistic: 0.0000, p-value: 0.0156



,year,EMR PHDI,Global PHDI
0,1990,66.0,68.0
1,1995,67.4,68.0
2,2000,67.2,70.0
3,2005,67.4,72.0
4,2010,68.5,72.0
5,2015,69.1,72.0
6,2018,69.6,73.0


In [61]:
stat, p, paired_data = analyze_wide_csvs(
    setting='emr vs emr',
    csv1_path=r'diet_calc\scores\phdi\total_males.csv',
    csv2_path=r'diet_calc\scores\phdi\total_females.csv',
    name1='EMR PHDI Males',
    name2='EMR PHDI Females',
    id_column='Country'
)
paired_data.round(1)

--- TEST SETTING: EMR VS EMR ---
Comparison: EMR PHDI Males Median vs EMR PHDI Females Median
Paired timepoints analyzed: 7 years
Wilcoxon statistic: 12.0000, p-value: 0.8125



,year,EMR PHDI Males,EMR PHDI Females
0,1990,66.0,65.4
1,1995,67.8,67.4
2,2000,66.9,67.5
3,2005,67.4,67.5
4,2010,68.2,68.8
5,2015,69.2,69.5
6,2018,69.7,69.6


**DASH Comparison**

In [62]:
stat, p, paired_data = analyze_wide_csvs(
    setting='emr vs global',
    csv1_path=r'diet_calc\scores\dash_dixon\total_all.csv',
    csv2_path=r'diet_calc\scores\dash_dixon\global\total_all.csv',
    name1='EMR DASH',
    name2='Global DASH',
    id_column='iso3'
)

paired_data.round(1)

--- TEST SETTING: EMR VS GLOBAL ---
Comparison: EMR DASH Median vs Global DASH Value
Paired timepoints analyzed: 7 years
Wilcoxon statistic: 0.0000, p-value: 0.0156



,year,EMR DASH,Global DASH
0,1990,1.4,1.0
1,1995,1.3,1.0
2,2000,1.4,1.0
3,2005,1.1,1.0
4,2010,1.2,1.0
5,2015,1.3,1.0
6,2018,1.4,1.0


In [63]:
stat, p, paired_data = analyze_wide_csvs(
    setting='emr vs emr',
    csv1_path=r'diet_calc\scores\dash_dixon\total_males.csv',
    csv2_path=r'diet_calc\scores\dash_dixon\total_females.csv',
    name1='EMR DASH Males',
    name2='EMR DASH Females',
    id_column='iso3'
)
paired_data.round(1)

--- TEST SETTING: EMR VS EMR ---
Comparison: EMR DASH Males Median vs EMR DASH Females Median
Paired timepoints analyzed: 7 years
Wilcoxon statistic: 1.0000, p-value: 0.0312



,year,EMR DASH Males,EMR DASH Females
0,1990,1.4,1.6
1,1995,1.4,1.3
2,2000,1.4,1.6
3,2005,1.0,1.7
4,2010,1.1,1.6
5,2015,1.3,1.7
6,2018,1.1,1.7


**Medit Comparison**

In [64]:
stat, p, paired_data = analyze_wide_csvs(
    setting='emr vs global',
    csv1_path=r'diet_calc\scores\medit_new\total_all.csv',
    csv2_path=r'diet_calc\scores\medit_new\global\total_all.csv',
    name1='EMR Medit',
    name2='Global Medit',
    id_column='iso3'
)

paired_data.round(1)

--- TEST SETTING: EMR VS GLOBAL ---
Comparison: EMR Medit Median vs Global Medit Value
Paired timepoints analyzed: 7 years
Wilcoxon statistic: 0.0000, p-value: 0.0156



,year,EMR Medit,Global Medit
0,1990,2.8,2
1,1995,2.6,2
2,2000,3.0,2
3,2005,2.9,2
4,2010,3.1,3
5,2015,3.1,2
6,2018,3.3,2


In [65]:
stat, p, paired_data = analyze_wide_csvs(
    setting='emr vs emr',
    csv1_path=r'diet_calc\scores\medit_new\total_males.csv',
    csv2_path=r'diet_calc\scores\medit_new\total_females.csv',
    name1='EMR Medit Males',
    name2='EMR Medit Females',
    id_column='iso3'
)

paired_data.round(1)

--- TEST SETTING: EMR VS EMR ---
Comparison: EMR Medit Males Median vs EMR Medit Females Median
Paired timepoints analyzed: 7 years
Wilcoxon statistic: 1.5000, p-value: 0.7500



,year,EMR Medit Males,EMR Medit Females
0,1990,2.8,2.8
1,1995,2.6,2.6
2,2000,2.9,2.9
3,2005,2.9,2.8
4,2010,3.0,3.1
5,2015,3.1,3.2
6,2018,3.3,3.3
